In [ ]:
import numpy as np
import pandas as pd

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import joblib
import warnings
import time
warnings.filterwarnings("ignore")
from sklearn.svm import SVR

from joblib import Parallel, delayed


In [ ]:
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"joblib version: {joblib.__version__}")

In [ ]:
df_train_production = pd.read_csv('df_train_production.csv')

In [ ]:
df_train_production.drop(columns = ['row_id'], inplace = True)
df_train_production.shape

In [ ]:
df_train_production.dropna(inplace=True)

In [ ]:
df_train_production['unit_target'] = df_train_production['target']/df_train_production['installed_capacity']
df_train_production['unit_target_48h'] = df_train_production['target_48h']/df_train_production['installed_capacity_48h']

In [ ]:
df_train_production.dropna(inplace=True)

In [ ]:
df_train_production.drop(columns=['year', 'is_consumption', 'day', 'installed_capacity', 'prediction_unit_id', 'target', 'eic_count'], inplace=True)

In [ ]:
df_train_production = df_train_production.sample(frac=0.01, random_state=42)

In [ ]:
X_prod = df_train_production.drop(columns=['unit_target', 'datetime'])
y_prod = df_train_production[['unit_target', 'datetime']]

split_index = int(0.6 * len(X_prod))

X_train_prod, X_test_prod = X_prod[:split_index], X_prod[split_index:]
y_train_prod, y_test_prod = y_prod[:split_index], y_prod[split_index:]

In [ ]:
top_features = ['surface_solar_radiation_downwards', 'surface_solar_radiation_downwards_min', 'is_business', 'unit_target_48h', 'total_precipitation_max', 'month', 'direct_solar_radiation_max', 'sin(hour)', 'product_type', 'total_precipitation', 'hour', 'is_country_holiday', 'cos(dayofyear)', 'total_precipitation_min', 'county', 'cloudcover_total_historical_grouped_by_date_48h', 'cloudcover_total_historical_mean_48h', 'target_168h', 'target_336h', 'cloudcover_low_max', 'cos(hour)', 'cloudcover_low_historical_grouped_by_date_48h', 'direct_solar_radiation', 'cloudcover_high_historical_mean_24h', 'installed_capacity_48h']

#Selecting the top N features from the dataset
X_train_prod_reduced = X_train_prod[top_features]
X_test_prod_reduced = X_test_prod[top_features]

GPR

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_prod_reduced)
X_test_scaled = scaler.transform(X_test_prod_reduced)

In [ ]:
def train_svr(X_train_chunk, y_train_chunk):
    model = SVR(kernel='rbf', gamma='auto', C=2**6, epsilon=2**-7)
    model.fit(X_train_chunk, y_train_chunk)
    return model

n_chunks = 512  
X_train_chunks = np.array_split(X_train_scaled, n_chunks)
y_train_chunks = np.array_split(y_train_prod['unit_target'], n_chunks)

start_time_tr = time.time()
svr_models = Parallel(n_jobs=-1)(delayed(train_svr)(X_chunk, y_chunk) for X_chunk, y_chunk in zip(X_train_chunks, y_train_chunks))
end_time_tr = time.time()

def parallel_predict(models, X):
    preds = np.mean([model.predict(X) for model in models], axis=0)
    return preds

X_test_chunks = np.array_split(X_test_scaled, n_chunks)

start_time_pr = time.time()
svr_pred_chunks = Parallel(n_jobs=-1)(delayed(parallel_predict)(svr_models, chunk) for chunk in X_test_chunks)
svr_pred = np.concatenate(svr_pred_chunks)
end_time_pr = time.time()

training_time = end_time_tr - start_time_tr
prediction_time = end_time_pr - start_time_pr

mae = mean_absolute_error(y_test_prod['unit_target'], svr_pred)
r2 = r2_score(y_test_prod['unit_target'], svr_pred)
rmse = np.sqrt(mean_squared_error(y_test_prod['unit_target'], svr_pred))
mse = mean_squared_error(y_test_prod['unit_target'], svr_pred)

results = {
    'Model': ['Parallel SVR (Fitting & Prediction)'],
    'Training Time (s)': [training_time],
    'Prediction Time (s)': [prediction_time],
    'MAE': [mae],
    'R-squared': [r2],
    'RMSE': [rmse],
    'MSE': [mse]
}

results_df = pd.DataFrame(results)
results_df.to_csv('parallel_svr_fitting_metrics.csv', index=False)

joblib.dump(svr_models, 'parallel_svr_models.joblib')

print("Metrics saved to 'parallel_svr_fitting_metrics.csv'")
print("Models saved as 'parallel_svr_models.joblib'")
print(f"Training time: {training_time:.4f} seconds")
print(f"Prediction time: {prediction_time:.4f} seconds")